In [5]:
import numpy as np
import scipy
import json
import matplotlib.pyplot as plt
from datetime import datetime
from qiskit.quantum_info import Pauli

np.set_printoptions(linewidth=200, precision=4, suppress=True)

In [6]:
# Initialize Pauli matrices

pauli_x = Pauli('X')
pauli_y = Pauli('Y')
pauli_z = Pauli('Z')
identity = Pauli('I')

X = pauli_x.to_matrix()
Y = pauli_y.to_matrix()
Z = pauli_z.to_matrix()
I = identity.to_matrix()

Ix1 = np.kron(X/2,I)
Iy1 = np.kron(Y/2,I)
Iz1 = np.kron(Z/2,I)
Ix2 = np.kron(I,X/2)
Iy2 = np.kron(I,Y/2)
Iz2 = np.kron(I,Z/2)

J = 696

In [7]:
U_cnot = np.array([[1,0,0,0],
                   [0,1,0,0],
                   [0,0,0,1],
                   [0,0,1,0]])

U_perm = np.array([[1,0,0,0],
                   [0,0,0,1],
                   [0,1,0,0],
                   [0,0,1,0]])

U_cz = np.array([[1,0,0,0],
                [0,1,0,0],
                [0,0,1,0],
                [0,0,0,-1]])

Unitary matrix --> Hamiltonian and pulse parameters = difficult

Pulse parameters --> Unitary matrix = easy

Method: change pulse parameters, calculate how similar the resulting Unitary matrix is to Target unitary matrix, compare and optimize

In [8]:
# Parameters for a 2 qubit Rx(pi/2) pulse, Hydrogen and Phosphorus

R1=R2=1
w_rf=6250
phi_rf=np.pi/2
t_rf=40e-6

w0=w_rf
w1_H=w1_P=6250


In [9]:
# 2 qubit Unitary matrix calculation from system Hamiltonian

def twoqubit_unitary(R1,R2,w_rf,phi_rf,t_rf,w0,w1_H,w1_P):
    
    H0 = 2*np.pi*(w0-w_rf)*Iz1+2*np.pi*(w0-w_rf)*Iz2

    HJ = 2*np.pi*J*(np.kron(Z,Z)/4)

    H1 = 2*np.pi*R1*w1_H*(np.cos(phi_rf)*Ix1+np.sin(phi_rf)*Iy1)+2*np.pi*R2*w1_P*(np.cos(phi_rf)*Ix2+np.sin(phi_rf)*Iy2)

    return scipy.linalg.expm(-1j*(H0+HJ+H1)*t_rf)



In [10]:
Rypi2 = twoqubit_unitary(R1,R2,w_rf,phi_rf,t_rf,w0,w1_H,w1_P)

In [11]:
rho = Iz1

rho_f = Rypi2 @ rho @ Rypi2.conj().T

rho_f

array([[ 0.0003+0.j    ,  0.    -0.j    ,  0.4993-0.0219j, -0.    -0.0139j],
       [ 0.    +0.j    ,  0.0003+0.j    , -0.    -0.0139j,  0.4993+0.0219j],
       [ 0.4993+0.0219j, -0.    +0.0139j, -0.0003+0.j    ,  0.    +0.j    ],
       [-0.    +0.0139j,  0.4993-0.0219j,  0.    -0.j    , -0.0003+0.j    ]])

In [12]:
print(np.trace(rho_f @ Ix1))
print(np.trace(rho_f @ Iy1))
print(np.trace(rho_f @ Iz1))


(0.9986568311717521+0j)
(-6.591949208711867e-17+0j)
(0.000608575287973024+0j)


1. Create unitary operator

$$
U = R_{-z}^2\left(\frac{\pi}{4}\right)
    R_{-y}^2\left(\frac{\pi}{4}\right)
    R_{z}^1\left(\frac{\pi}{4}\right)
    R_{y}^1\left(\frac{\pi}{4}\right)
$$

In [13]:
# Rotation Rz through euler angles Rz(theta)=Rx(pi/2)Ry(theta)Rx(-pi/2)

R1 = twoqubit_unitary(0,1,6250,0,40e-6,6250,6250,6250)
R2 = twoqubit_unitary(0,1,6250,-np.pi/2,20e-6,6250,6250,6250)
R3 = twoqubit_unitary(0,1,6250,np.pi,40e-6,6250,6250,6250)

R2_zpi4 = R1 @ R2 @ R3

print(R2_zpi4)



[[ 0.9409+0.3308j  0.0729-0.j      0.    +0.j      0.    +0.j    ]
 [-0.0729-0.j      0.9409-0.3308j  0.    +0.j      0.    +0.j    ]
 [ 0.    +0.j      0.    +0.j      0.8983+0.4334j -0.0724-0.j    ]
 [ 0.    +0.j      0.    +0.j      0.0724+0.j      0.8983-0.4334j]]


In [14]:
# Rotation Ry

R2_ypi4 = twoqubit_unitary(0,1,6250,-np.pi/2,20e-6,6250,6250,6250)

In [15]:
# Rotation Rz through euler angles Rz(theta)=Rx(pi/2)Ry(theta)Rx(-pi/2)

R1 = twoqubit_unitary(1,0,6250,0,40e-6,6250,6250,6250)
R2 = twoqubit_unitary(1,0,6250,np.pi/2,20e-6,6250,6250,6250)
R3 = twoqubit_unitary(1,0,6250,np.pi,40e-6,6250,6250,6250)

R1zpi4 = R1 @ R2 @ R3

print(R1zpi4)


[[ 0.8983-0.4334j  0.    +0.j      0.0724-0.j      0.    +0.j    ]
 [ 0.    +0.j      0.9409-0.3308j  0.    +0.j     -0.0729-0.j    ]
 [-0.0724+0.j      0.    +0.j      0.8983+0.4334j  0.    +0.j    ]
 [ 0.    +0.j      0.0729-0.j      0.    +0.j      0.9409+0.3308j]]


In [16]:
# Rotation Ry

R1ypi4 = twoqubit_unitary(1,0,6250,np.pi/2,20e-6,6250,6250,6250)

U achieved using hard pulses

In [17]:
U = R2_zpi4 @ R2_ypi4 @ R1zpi4 @ R1ypi4

print(U)

[[ 0.8392-0.1136j  0.3991-0.0133j -0.2826+0.0629j -0.1991-0.j    ]
 [-0.3088+0.2879j  0.6379-0.4843j  0.0969-0.107j  -0.3259+0.2307j]
 [ 0.1646+0.2581j  0.0969+0.107j   0.5435+0.7115j  0.1871+0.221j ]
 [-0.1047-0.j      0.4162-0.0706j -0.305 +0.0258j  0.8392-0.1136j]]


Strongly Modulated Pulse optimization

In [14]:

# SMP following SpinQ manual 

# constants
n = 2
w0 = 6250
w1_max = 6250

# Compute fidelity aka loss function

def fidelity_func_smp(R_H, R_P, phi_H, phi_P, dt, w1_H, w1_P, U_target):

    U = np.eye(4, dtype=complex)

    HJ = 2*np.pi*J*(np.kron(Z,Z)/4)

    for k in range(len(R_H)):

        Hrf = (
            2*np.pi*R_H[k]*w1_H*(np.cos(phi_H[k])*Ix1 + np.sin(phi_H[k])*Iy1)
            +
            2*np.pi*R_P[k]*w1_P*(np.cos(phi_P[k])*Ix2 + np.sin(phi_P[k])*Iy2)
        )

        H = HJ + Hrf

        U = scipy.linalg.expm(-1j * H * dt) @ U

    d = U_target.shape[0]

    return 1 - np.abs(np.trace(U_target.conj().T @ U)) / d

# Gradient descent algorithm

# Calculate numerical gradient

def num_grad(
        R_H, R_P,
        phi_H, phi_P,
        N, dt,
        w1_H, w1_P,
        U_target,
        d=1e-2):
    
    grad_R_H = np.zeros_like(R_H)
    grad_R_P = np.zeros_like(R_P)
    grad_phi_H = np.zeros_like(phi_H)
    grad_phi_P = np.zeros_like(phi_P)

    eps = 1e-6
    # per-component increments
    dR_H = np.maximum(d*R_H, eps)
    dR_P = np.maximum(d*R_P, eps)
    dphi_H = np.maximum(d*np.abs(phi_H), eps)
    dphi_P = np.maximum(d*np.abs(phi_P), eps)

    Q0 = fidelity_func_smp(R_H, R_P, phi_H, phi_P, dt, w1_H, w1_P, U_target)

    for k in range(N):
        R_H_temp = R_H.copy()
        R_H_temp[k] += dR_H[k]
        grad_R_H[k] = (fidelity_func_smp(R_H_temp, R_P, phi_H, phi_P, dt, w1_H, w1_P, U_target) - Q0)/dR_H[k]
        
        R_P_temp = R_P.copy()
        R_P_temp[k] += dR_P[k]
        grad_R_P[k] = (fidelity_func_smp(R_H, R_P_temp, phi_H, phi_P, dt, w1_H, w1_P, U_target) - Q0)/dR_P[k]
        
        phi_H_temp = phi_H.copy()
        phi_H_temp[k] += dphi_H[k]
        grad_phi_H[k] = (fidelity_func_smp(R_H, R_P, phi_H_temp, phi_P, dt, w1_H, w1_P, U_target) - Q0)/dphi_H[k]

        phi_P_temp = phi_P.copy()
        phi_P_temp[k] += dphi_P[k]
        grad_phi_P[k] = (fidelity_func_smp(R_H, R_P, phi_H, phi_P_temp, dt, w1_H, w1_P, U_target) - Q0)/dphi_P[k]


    return grad_R_H, grad_R_P, grad_phi_H, grad_phi_P

# Save to json file

def export_to_json_SMP(filename, title, fidelity, totalpulsewidth, slices, R, phi, omega_rf, owner="Unkown"):
    
    R = np.asarray(R).flatten()
    phi = np.asarray(phi).flatten()
    omega_rf = np.asarray(omega_rf).flatten()
    
    dt = totalpulsewidth / slices

    channel1_pulses = []
    channel2_pulses = []

    for k in range(slices):
        
        amplitude = 100 * float(R[k]) # in %
        phase = float(np.degrees(phi[k]) % 360)
        detuning = float(w0 - omega_rf[k])

        pulse_entry = {
            "detuning": detuning,
            "phase": phase,
            "amplitude": amplitude,
            "width": float(dt)
        }

        # Same pulse on both channels (modify if needed)
        channel1_pulses.append(pulse_entry)
        channel2_pulses.append(pulse_entry.copy())
        
        data = {
        "description": {
            "TITLE": title,
            "OWNER": owner,
            "DATE": datetime.now().strftime("%d-%m-%Y"),
            "FIDELITY": float(fidelity),
            "TOTALPULSEWIDTH": float(totalpulsewidth),
            "SLICES": int(slices)
        },
        "parameters": {
            "offset": [
                {
                    "channel1_pulsefre_offset": 0,
                    "channel1_framefre_offset": 0,
                    "channel2_pulsefre_offset": 0,
                    "channel2_framefre_offset": 0
                }
            ]
        },
        "pulse": {
            "channel1_pulse": channel1_pulses,
            "channel2_pulse": channel2_pulses
        }
    }

    with open(filename, "w") as f:
        json.dump(data, f, indent=4)

    print(f"Pulse file saved to {filename}")


In [15]:
# SMP Gradient descent loop

U_target = U

J = 696
T = 1e-3
N = 500  # maybe add more divisions?
dt = T/N
w1_H = w1_P = 6250

# initialize
R_H = np.full(N, 1.0, dtype=float)
R_P = np.full(N, 1.0, dtype=float)
phi_H = np.zeros(N, dtype=float)
phi_P = np.zeros(N, dtype=float)

learn_rate = 0.05
iter = 200

for i in range(iter):
    grad_R_H, grad_R_P, grad_phi_H, grad_phi_P = num_grad(
    R_H, R_P,
    phi_H, phi_P,
    N, dt,
    w1_H, w1_P,
    U_target,
    d=0.02
)
    
    R_H -= learn_rate * grad_R_H
    R_P -= learn_rate * grad_R_P
    phi_H -= learn_rate * grad_phi_H
    phi_P -= learn_rate * grad_phi_P
    
    R_H = np.clip(R_H,0,1)
    R_P = np.clip(R_P,0,1)
    phi_H = phi_H % (2*np.pi)
    phi_P = phi_P % (2*np.pi)
    
    if i % 10 == 0 or i == iter-1:
        print(f"Iteration {i}, Fidelity={1-fidelity_func_smp(R_H, R_P, phi_H, phi_P, dt, w1_H, w1_P, U_target):.6f}")
        
pulse_fidelity = 1-fidelity_func_smp(R_H, R_P, phi_H, phi_P, dt, w1_H, w1_P, U_target)

Iteration 0, Fidelity=0.143218
Iteration 10, Fidelity=0.211247
Iteration 20, Fidelity=0.312089
Iteration 30, Fidelity=0.423404
Iteration 40, Fidelity=0.523587
Iteration 50, Fidelity=0.601875
Iteration 60, Fidelity=0.657940
Iteration 70, Fidelity=0.696445
Iteration 80, Fidelity=0.722649
Iteration 90, Fidelity=0.740667
Iteration 100, Fidelity=0.753289
Iteration 110, Fidelity=0.762309
Iteration 120, Fidelity=0.768867
Iteration 130, Fidelity=0.773700
Iteration 140, Fidelity=0.777300
Iteration 150, Fidelity=0.780003
Iteration 160, Fidelity=0.782047
Iteration 170, Fidelity=0.783603
Iteration 180, Fidelity=0.784796
Iteration 190, Fidelity=0.785720
Iteration 199, Fidelity=0.786377


In [25]:
export_to_json_SMP("TestSMP_hardpulses","pulses1",pulse_fidelity,T,N,R,phi,omega_rf,"andreroque")

Pulse file saved to TestSMP_hardpulses


In [36]:
U_target = U_cnot

J = 696
T = 200e-6
N = 50  # maybe add more divisions?
dt = T/N

# initialize
R = np.full(N, 1.0, dtype=float)
phi = np.zeros(N, dtype=float)
omega_rf = np.full(N, w0, dtype=float)
dts = np.full(N, dt, dtype=float)

learn_rate = 0.05 # 500 iterations 0.01 is small - 38 minutes - fidelity = 0.7244; 500 iterations 0.05 - plateau at fidelity = 0.74
iter = 200

for i in range(iter):
    grad_R,grad_phi,grad_omega = num_grad(R, phi, omega_rf, dts, U_target) #,grad_dt
    
    R -= learn_rate * grad_R
    phi -= learn_rate * grad_phi
    omega_rf -= learn_rate * grad_omega
    #dts -= learn_rate * grad_dt
    
    R = np.clip(R,0,1)
    phi = phi % (2*np.pi)
    
    if i % 10 == 0 or i == iter-1:
        print(f"Iteration {i}, Fidelity={fidelity(R, phi, omega_rf, dts, U_target):.6f}")

Iteration 0, Fidelity=0.352359
Iteration 10, Fidelity=0.385224
Iteration 20, Fidelity=0.412809
Iteration 30, Fidelity=0.435352
Iteration 40, Fidelity=0.453416
Iteration 50, Fidelity=0.467692
Iteration 60, Fidelity=0.478872
Iteration 70, Fidelity=0.487583
Iteration 80, Fidelity=0.494357
Iteration 90, Fidelity=0.499630
Iteration 100, Fidelity=0.503748
Iteration 110, Fidelity=0.506982
Iteration 120, Fidelity=0.509543
Iteration 130, Fidelity=0.511589
Iteration 140, Fidelity=0.513244
Iteration 150, Fidelity=0.514599
Iteration 160, Fidelity=0.515724
Iteration 170, Fidelity=0.516671
Iteration 180, Fidelity=0.517480
Iteration 190, Fidelity=0.518180
Iteration 199, Fidelity=0.518737


GRAPE Algorithm

In [ ]:
# Gradient Ascent - GRAPE - same channel

# a bit different from the basic SMP, the parameter phi is omitted, collapsed into the amplitude which now differs for each orthogonal direction x, y

# constants
n = 2
d = 2**n
#wH = 27e6
#wP = 11e6
#w1H = w1P after calibration
w1_max = 6250

H_x = 2*np.pi*w1_max*(Ix1+Ix2)
H_y = 2*np.pi*w1_max*(Iy1+Iy2)

# H_0 = 0 because on resonance, only include when detuning or counteracting chemical shift mismatch
H_J = 2*np.pi*J*(np.kron(Z,Z)/4)

def p_calc(R_x, R_y, U_target):
    P_list = []
    P = U_target.copy()
    
    for k in reversed(range(N)):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        P = Uk.conj().T @ P
        P_list.insert(0, P)

    return P_list

def x_calc(R_x,R_y):
    X_list = []
    U = np.eye(d, dtype=complex)
    
    for k in range(N):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        U = Uk @ U
        X_list.append(U)

    return X_list, U

def fidelity_grape(U):
    return np.abs(np.trace(U_target.conj().T @ U))/d

def grape_grad(R_x, R_y, U_target):
    
    X_list, U_final = x_calc(R_x, R_y)
    P_list = p_calc(R_x, R_y, U_target)
    
    H_x = 2*np.pi*w1_max*(Ix1+Ix2)
    H_y = 2*np.pi*w1_max*(Iy1+Iy2)
    H_J = 2*np.pi*J*(np.kron(Z,Z)/4)
    
    phi = np.trace(U_target.conj().T @ U_final)
    F0 = fidelity_grape(U_final)
    
    grad_x = np.zeros((N,1))
    grad_y = np.zeros((N,1))
    
    for k in range(N):
        Xk = X_list[k]
        Pk = P_list[k]
        
        dX = -1j * dt * H_x @ Xk
        dY = -1j * dt * H_y @ Xk
        
        grad_x[k] = (2/d) * np.real(
            np.trace(Pk.conj().T @ dX) * np.conj(phi)
        )
        
        grad_y[k] = (2/d) * np.real(
            np.trace(Pk.conj().T @ dY) * np.conj(phi)
        )
        
    return F0, grad_x, grad_y, U_final

def export_to_json_GRAPE(filename, title, fidelity, totalpulsewidth, slices, R_x, R_y, owner="Unkown"):
    
    R_x = np.asarray(R_x).flatten()
    R_y = np.asarray(R_y).flatten()
    
    dt = totalpulsewidth / slices

    channel1_pulses = []
    channel2_pulses = []

    for k in range(slices):

        rx = float(R_x[k])
        ry = float(R_y[k])

        amplitude = 100 * np.sqrt(rx**2 + ry**2) # in %
        phase = np.degrees(np.arctan2(ry, rx))  # convert to degrees

        pulse_entry = {
            "detuning": 0,
            "phase": float(phase),
            "amplitude": float(amplitude),
            "width": float(dt)
        }

        # Same pulse on both channels (modify if needed)
        channel1_pulses.append(pulse_entry)
        channel2_pulses.append(pulse_entry.copy())
        
        data = {
        "description": {
            "TITLE": title,
            "OWNER": owner,
            "DATE": datetime.now().strftime("%d-%m-%Y"),
            "FIDELITY": float(fidelity),
            "TOTALPULSEWIDTH": float(totalpulsewidth),
            "SLICES": int(N)
        },
        "parameters": {
            "offset": [
                {
                    "channel1_pulsefre_offset": 0,
                    "channel1_framefre_offset": 0,
                    "channel2_pulsefre_offset": 0,
                    "channel2_framefre_offset": 0
                }
            ]
        },
        "pulse": {
            "channel1_pulse": channel1_pulses,
            "channel2_pulse": channel2_pulses
        }
    }

    with open(filename, "w") as f:
        json.dump(data, f, indent=4)

    print(f"Pulse file saved to {filename}")
    

In [18]:
# Gradient Ascent - GRAPE - different channels

# a bit different from the basic SMP, the parameter phi is omitted, collapsed into the amplitude which now differs for each orthogonal direction x, y

# constants
n = 2
d = 2**n
#wH = 27e6
#wP = 11e6
#w1H = w1P after calibration
w1_max = 6250

H_x_H = 2*np.pi*w1_max*(Ix1)
H_x_P = 2*np.pi*w1_max*(Ix2)
H_y_H = 2*np.pi*w1_max*(Iy1)
H_y_P = 2*np.pi*w1_max*(Iy2)

# H_0 = 0 because on resonance, only include when detuning or counteracting chemical shift mismatch
H_J = 2*np.pi*J*(np.kron(Z,Z)/4)

def p_calc_2channel(Rx_H, Ry_H, Rx_P, Ry_P, U_target):
    P_list = []
    P = U_target.copy()
    
    for k in reversed(range(N)):
        H = H_J + Rx_H[k]*H_x_H + Ry_H[k]*H_y_H + Rx_P[k]*H_x_P + Ry_P[k]*H_y_P
        Uk = scipy.linalg.expm(-1j * H * dt)
        P = Uk.conj().T @ P
        P_list.insert(0, P)

    return P_list

def x_calc_2channel(Rx_H, Ry_H, Rx_P, Ry_P):
    X_list = []
    U = np.eye(d, dtype=complex)
    
    for k in range(N):
        H = H_J + Rx_H[k]*H_x_H + Ry_H[k]*H_y_H + Rx_P[k]*H_x_P + Ry_P[k]*H_y_P
        Uk = scipy.linalg.expm(-1j * H * dt)
        U = Uk @ U
        X_list.append(U)

    return X_list, U

def fidelity_grape(U, U_target):
    phi = np.trace(U_target.conj().T @ U)
    return np.abs(phi)**2/d**2

def grape_grad_2channel(Rx_H, Ry_H, Rx_P, Ry_P, U_target):
    
    X_list, U_final = x_calc_2channel(Rx_H, Ry_H, Rx_P, Ry_P)
    P_list = p_calc_2channel(Rx_H, Ry_H, Rx_P, Ry_P, U_target)
    
    phi = np.trace(U_target.conj().T @ U_final)
    F0 = fidelity_grape(U_final, U_target)
    
    grad_Rx_H = np.zeros(N)
    grad_Ry_H = np.zeros(N)
    grad_Rx_P = np.zeros(N)
    grad_Ry_P = np.zeros(N)
    
    for k in range(N):
        Xk = X_list[k]
        Pk = P_list[k]
        
        dX_Rx_H = -1j * dt * H_x_H @ Xk
        dX_Ry_H = -1j * dt * H_y_H @ Xk
        dX_Rx_P = -1j * dt * H_x_P @ Xk
        dX_Ry_P = -1j * dt * H_y_P @ Xk

        grad_Rx_H[k] = 2 * np.real(
            np.trace(Pk.conj().T @ dX_Rx_H) * np.conj(phi)
        )

        grad_Ry_H[k] = 2 * np.real(
            np.trace(Pk.conj().T @ dX_Ry_H) * np.conj(phi)
        )

        grad_Rx_P[k] = 2 * np.real(
            np.trace(Pk.conj().T @ dX_Rx_P) * np.conj(phi)
        )

        grad_Ry_P[k] = 2 * np.real(
            np.trace(Pk.conj().T @ dX_Ry_P) * np.conj(phi)
        )
        
    return (F0, grad_Rx_H, grad_Ry_H, grad_Rx_P, grad_Ry_P, U_final)

def export_to_json_GRAPE_2channel(filename, title, fidelity, totalpulsewidth, slices, Rx_H, Ry_H, Rx_P, Ry_P, owner="Unkown"):
    
    Rx_H = np.asarray(Rx_H).flatten()
    Ry_H = np.asarray(Ry_H).flatten()
    Rx_P = np.asarray(Rx_P).flatten()
    Ry_P = np.asarray(Ry_P).flatten()
    
    dt = totalpulsewidth / slices

    channel1_pulses = []
    channel2_pulses = []

    for k in range(slices):

        amplitude_H = 100 * np.sqrt(Rx_H[k]**2 + Ry_H[k]**2)  # percent
        phase_H = np.degrees(np.arctan2(Ry_H[k], Rx_H[k])) % 360

        amplitude_P = 100 * np.sqrt(Rx_P[k]**2 + Ry_P[k]**2)
        phase_P = np.degrees(np.arctan2(Ry_P[k], Rx_P[k])) % 360
        
        pulse_H = {
            "detuning": 0,
            "phase": float(phase_H),
            "amplitude": float(amplitude_H),
            "width": float(dt)
        }
        pulse_P = {
            "detuning": 0,
            "phase": float(phase_P),
            "amplitude": float(amplitude_P),
            "width": float(dt)
        }

        # Same pulse on both channels (modify if needed)
        channel1_pulses.append(pulse_H)
        channel2_pulses.append(pulse_P)
        
        data = {
        "description": {
            "TITLE": title,
            "OWNER": owner,
            "DATE": datetime.now().strftime("%d-%m-%Y"),
            "FIDELITY": float(fidelity),
            "TOTALPULSEWIDTH": float(totalpulsewidth),
            "SLICES": int(N)
        },
        "parameters": {
            "offset": [
                {
                    "channel1_pulsefre_offset": 0,
                    "channel1_framefre_offset": 0,
                    "channel2_pulsefre_offset": 0,
                    "channel2_framefre_offset": 0
                }
            ]
        },
        "pulse": {
            "channel1_pulse": channel1_pulses,
            "channel2_pulse": channel2_pulses
        }
    }

    with open(filename, "w") as f:
        json.dump(data, f, indent=4)

    print(f"Pulse file saved to {filename}")
    

In [31]:
# learning loop

np.random.seed(1234)
U_target = U_cnot
print("Target matrix: \n", U_target)

J = 696
T = 100e-3
N = 5000  # add more divisions if the fidelity doesnt increase monotonically
dt = T/N

print("dt = ",dt)

# initialize
R_x = 0.2 * np.random.rand(N,1)
R_y = 0.2 * np.random.rand(N,1)

learn_rate = 0.01
max_iter = 5000
fid_history = []

for i in range(max_iter):

    F, grad_x, grad_y, Ufinal = grape_grad(R_x, R_y, U_target)

    # update
    R_x_new = R_x + learn_rate * grad_x
    R_y_new = R_y + learn_rate * grad_y

    R_x_new = np.clip(R_x_new, -1, 1)
    R_y_new = np.clip(R_y_new, -1, 1)
    
    R_x = R_x_new
    R_y = R_y_new

    fid_history.append(F)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")

    if F > 0.999:
        print("Converged.")
        break

print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)
print(R_x, R_y)

Target matrix: 
 [[1 0 0 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]]
dt =  2e-05
Iter 0, Fidelity = 0.084315, lr = 1.000000e-02
Iter 100, Fidelity = 0.344374, lr = 1.000000e-02
Iter 200, Fidelity = 0.434962, lr = 1.000000e-02
Iter 300, Fidelity = 0.387223, lr = 1.000000e-02


KeyboardInterrupt: 

In [19]:
# learning loop --- CNOT

np.random.seed(1234)
U_target = U_cnot
print("Target matrix: \n", U_target)

J = 696
T = 1e-3    # 1ms so J coupling spin-spin interactions have time to evolve
N = 500     # add more divisions if the fidelity doesnt increase monotonically. dt has to be small enough
dt = T/N

print("dt = ",dt)

# initialize
Rx_H = 0.2 * np.random.rand(N)
Ry_H = 0.2 * np.random.rand(N)
Rx_P = 0.2 * np.random.rand(N)
Ry_P = 0.2 * np.random.rand(N)

learn_rate = 0.01
max_iter = 5000

for i in range(max_iter):

    F, gRxH, gRyH, gRxP, gRyP, Ufinal = grape_grad_2channel(
        Rx_H, Ry_H, Rx_P, Ry_P, U_target
    )

    Rx_H += learn_rate * gRxH
    Ry_H += learn_rate * gRyH
    Rx_P += learn_rate * gRxP
    Ry_P += learn_rate * gRyP

    # Clip amplitudes
    Rx_H = np.clip(Rx_H, -1, 1)
    Ry_H = np.clip(Ry_H, -1, 1)
    Rx_P = np.clip(Rx_P, -1, 1)
    Ry_P = np.clip(Ry_P, -1, 1)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")
        
    if F > 0.999:
        print("Converged.")
        break
    
print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)


Target matrix: 
 [[1 0 0 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]]
dt =  2e-06
Iter 0, Fidelity = 0.102051, lr = 1.000000e-02
Iter 100, Fidelity = 0.822771, lr = 1.000000e-02
Iter 200, Fidelity = 0.885421, lr = 1.000000e-02
Iter 300, Fidelity = 0.917422, lr = 1.000000e-02
Iter 400, Fidelity = 0.936242, lr = 1.000000e-02
Iter 500, Fidelity = 0.948115, lr = 1.000000e-02
Iter 600, Fidelity = 0.956094, lr = 1.000000e-02
Iter 700, Fidelity = 0.961739, lr = 1.000000e-02
Iter 800, Fidelity = 0.965897, lr = 1.000000e-02
Iter 900, Fidelity = 0.969056, lr = 1.000000e-02
Iter 1000, Fidelity = 0.971518, lr = 1.000000e-02
Iter 1100, Fidelity = 0.973475, lr = 1.000000e-02
Iter 1200, Fidelity = 0.975057, lr = 1.000000e-02
Iter 1300, Fidelity = 0.976356, lr = 1.000000e-02
Iter 1400, Fidelity = 0.977434, lr = 1.000000e-02
Iter 1500, Fidelity = 0.978339, lr = 1.000000e-02
Iter 1600, Fidelity = 0.979106, lr = 1.000000e-02
Iter 1700, Fidelity = 0.979762, lr = 1.000000e-02
Iter 1800, Fidelity = 0.980328, lr = 1

In [45]:
export_to_json_GRAPE_2channel("TestGRAPE_cnot","1stCNOT",F,T,N,Rx_H,Ry_H,Rx_P,Ry_P,"andreroque")

Pulse file saved to TestGRAPE_cnot


In [20]:
# learning loop --- Exercise Rotations

# NOTES:
# 

np.random.seed(1234)
U_target = U
print("Target matrix: \n", U_target)

J = 696
T = 1e-3    # 1ms so J coupling spin-spin interactions have time to evolve
N = 500     # add more divisions if the fidelity doesnt increase monotonically. dt has to be small enough
dt = T/N

print("dt = ",dt)

# initialize
Rx_H = 0.2 * np.random.rand(N)
Ry_H = 0.2 * np.random.rand(N)
Rx_P = 0.2 * np.random.rand(N)
Ry_P = 0.2 * np.random.rand(N)

learn_rate = 0.01
max_iter = 5000

for i in range(max_iter):

    F, gRxH, gRyH, gRxP, gRyP, Ufinal = grape_grad_2channel(
        Rx_H, Ry_H, Rx_P, Ry_P, U_target
    )

    Rx_H += learn_rate * gRxH
    Ry_H += learn_rate * gRyH
    Rx_P += learn_rate * gRxP
    Ry_P += learn_rate * gRyP

    # Clip amplitudes
    Rx_H = np.clip(Rx_H, -1, 1)
    Ry_H = np.clip(Ry_H, -1, 1)
    Rx_P = np.clip(Rx_P, -1, 1)
    Ry_P = np.clip(Ry_P, -1, 1)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")
        
    if F > 0.999:
        print("Converged.")
        break
    
print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)


Target matrix: 
 [[ 0.8392-0.1136j  0.3991-0.0133j -0.2826+0.0629j -0.1991-0.j    ]
 [-0.3088+0.2879j  0.6379-0.4843j  0.0969-0.107j  -0.3259+0.2307j]
 [ 0.1646+0.2581j  0.0969+0.107j   0.5435+0.7115j  0.1871+0.221j ]
 [-0.1047-0.j      0.4162-0.0706j -0.305 +0.0258j  0.8392-0.1136j]]
dt =  2e-06
Iter 0, Fidelity = 0.325959, lr = 1.000000e-02
Iter 100, Fidelity = 0.849218, lr = 1.000000e-02
Iter 200, Fidelity = 0.960366, lr = 1.000000e-02
Iter 300, Fidelity = 0.975567, lr = 1.000000e-02
Iter 400, Fidelity = 0.982338, lr = 1.000000e-02
Iter 500, Fidelity = 0.986502, lr = 1.000000e-02
Iter 600, Fidelity = 0.989297, lr = 1.000000e-02
Iter 700, Fidelity = 0.991259, lr = 1.000000e-02
Iter 800, Fidelity = 0.992674, lr = 1.000000e-02
Iter 900, Fidelity = 0.993712, lr = 1.000000e-02
Iter 1000, Fidelity = 0.994484, lr = 1.000000e-02
Iter 1100, Fidelity = 0.995064, lr = 1.000000e-02
Iter 1200, Fidelity = 0.995504, lr = 1.000000e-02
Iter 1300, Fidelity = 0.995841, lr = 1.000000e-02
Iter 1400, Fid

In [47]:
# learning loop --- Hadamard

# NOTES:
# 

np.random.seed(1234)
U_target = 0.5* np.array([[1,1,1,1],
                        [1,-1,1,-1],
                        [1,1,-1,-1],
                        [1,-1,-1,1]])
print("Target matrix: \n", U_target)

J = 696
T = 1e-3    # 1ms so J coupling spin-spin interactions have time to evolve
N = 500     # add more divisions if the fidelity doesnt increase monotonically. dt has to be small enough
dt = T/N

print("dt = ",dt)

# initialize
Rx_H = 0.2 * np.random.rand(N)
Ry_H = 0.2 * np.random.rand(N)
Rx_P = 0.2 * np.random.rand(N)
Ry_P = 0.2 * np.random.rand(N)

learn_rate = 0.01
max_iter = 5000

for i in range(max_iter):

    F, gRxH, gRyH, gRxP, gRyP, Ufinal = grape_grad_2channel(
        Rx_H, Ry_H, Rx_P, Ry_P, U_target
    )

    Rx_H += learn_rate * gRxH
    Ry_H += learn_rate * gRyH
    Rx_P += learn_rate * gRxP
    Ry_P += learn_rate * gRyP

    # Clip amplitudes
    Rx_H = np.clip(Rx_H, -1, 1)
    Ry_H = np.clip(Ry_H, -1, 1)
    Rx_P = np.clip(Rx_P, -1, 1)
    Ry_P = np.clip(Ry_P, -1, 1)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")
        
    if F > 0.999:
        print("Converged.")
        break
    
print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)


Target matrix: 
 [[ 0.5  0.5  0.5  0.5]
 [ 0.5 -0.5  0.5 -0.5]
 [ 0.5  0.5 -0.5 -0.5]
 [ 0.5 -0.5 -0.5  0.5]]
dt =  2e-06
Iter 0, Fidelity = 0.103521, lr = 1.000000e-02
Iter 100, Fidelity = 0.211560, lr = 1.000000e-02
Iter 200, Fidelity = 0.318850, lr = 1.000000e-02
Iter 300, Fidelity = 0.401317, lr = 1.000000e-02
Iter 400, Fidelity = 0.460506, lr = 1.000000e-02
Iter 500, Fidelity = 0.498963, lr = 1.000000e-02
Iter 600, Fidelity = 0.525417, lr = 1.000000e-02
Iter 700, Fidelity = 0.545346, lr = 1.000000e-02
Iter 800, Fidelity = 0.561441, lr = 1.000000e-02
Iter 900, Fidelity = 0.575055, lr = 1.000000e-02
Iter 1000, Fidelity = 0.586938, lr = 1.000000e-02
Iter 1100, Fidelity = 0.597545, lr = 1.000000e-02
Iter 1200, Fidelity = 0.607167, lr = 1.000000e-02
Iter 1300, Fidelity = 0.615994, lr = 1.000000e-02
Iter 1400, Fidelity = 0.624147, lr = 1.000000e-02
Iter 1500, Fidelity = 0.631705, lr = 1.000000e-02
Iter 1600, Fidelity = 0.638718, lr = 1.000000e-02
Iter 1700, Fidelity = 0.645221, lr = 1.0

In [21]:
# learning loop --- BELL STATE

# NOTES:
# needs more than 1ms, fidelity is stuck at 0.25

np.random.seed(1234)
U_target = np.array([[1,0,0,1],
                    [0,0,0,0],
                    [0,0,0,0],
                    [1,0,0,1]])
print("Target matrix: \n", U_target)

J = 696
T = 2e-3    # 1ms so J coupling spin-spin interactions have time to evolve
N = 1000     # add more divisions if the fidelity doesnt increase monotonically. dt has to be small enough
dt = T/N

print("dt = ",dt)

# initialize
Rx_H = 0.2 * np.random.rand(N)
Ry_H = 0.2 * np.random.rand(N)
Rx_P = 0.2 * np.random.rand(N)
Ry_P = 0.2 * np.random.rand(N)

learn_rate = 0.01
max_iter = 5000

for i in range(max_iter):

    F, gRxH, gRyH, gRxP, gRyP, Ufinal = grape_grad_2channel(
        Rx_H, Ry_H, Rx_P, Ry_P, U_target
    )

    Rx_H += learn_rate * gRxH
    Ry_H += learn_rate * gRyH
    Rx_P += learn_rate * gRxP
    Ry_P += learn_rate * gRyP

    # Clip amplitudes
    Rx_H = np.clip(Rx_H, -1, 1)
    Ry_H = np.clip(Ry_H, -1, 1)
    Rx_P = np.clip(Rx_P, -1, 1)
    Ry_P = np.clip(Ry_P, -1, 1)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")
        
    if F > 0.999:
        print("Converged.")
        break
    
print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)


Target matrix: 
 [[1 0 0 1]
 [0 0 0 0]
 [0 0 0 0]
 [1 0 0 1]]
dt =  2e-06
Iter 0, Fidelity = 0.019008, lr = 1.000000e-02
Iter 100, Fidelity = 0.249971, lr = 1.000000e-02
Iter 200, Fidelity = 0.249991, lr = 1.000000e-02
Iter 300, Fidelity = 0.249991, lr = 1.000000e-02
Iter 400, Fidelity = 0.249991, lr = 1.000000e-02
Iter 500, Fidelity = 0.249991, lr = 1.000000e-02
Iter 600, Fidelity = 0.249991, lr = 1.000000e-02
Iter 700, Fidelity = 0.249991, lr = 1.000000e-02
Iter 800, Fidelity = 0.249991, lr = 1.000000e-02
Iter 900, Fidelity = 0.249991, lr = 1.000000e-02
Iter 1000, Fidelity = 0.249991, lr = 1.000000e-02
Iter 1100, Fidelity = 0.249991, lr = 1.000000e-02
Iter 1200, Fidelity = 0.249991, lr = 1.000000e-02
Iter 1300, Fidelity = 0.249991, lr = 1.000000e-02
Iter 1400, Fidelity = 0.249991, lr = 1.000000e-02
Iter 1500, Fidelity = 0.249991, lr = 1.000000e-02
Iter 1600, Fidelity = 0.249991, lr = 1.000000e-02
Iter 1700, Fidelity = 0.249991, lr = 1.000000e-02
Iter 1800, Fidelity = 0.249991, lr = 1